# 🌍 Indexes — Data Collector
**Data source:** Yahoo Finance &nbsp;·&nbsp; `yfinance` Python library

This notebook lets you download price data for major **stock market indexes**.

Indexes track the performance of a basket of stocks and are used as benchmarks
to compare individual stock or portfolio performance.

### What you can download
| Category | What it contains |
|---|---|
| 📊 Prices | Adjusted close price history |
| 📈 Returns | Period-over-period percentage returns |
| 📈 Cumulative Returns | Total return since start of the period |

### How to use this notebook
1. **Run the Setup cell** (`Shift + Enter`) — do this first
2. Select indexes from the preset list and/or add custom ones
3. Set the date range and frequency
4. Click **Download Data**

The output is saved to the `data/` folder. If you use the same file name as
the Stocks or Macro notebooks, index data will appear as additional sheets
in the same Excel file.


In [ ]:
# ── Setup — Run this cell first ─────────────────────────────────────────────
import sys
import datetime
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

from src.collect import download_prices
from src.transform import resample_prices, calculate_returns, calculate_cumulative_returns
from src.export import save_to_excel

print("✅ Setup complete — continue to the next cell.")


---
## Step 1 — Select Indexes

Choose from the preset list below. You can also add any index manually using
its Yahoo Finance ticker (e.g. `^NSEI` for India's Nifty 50).

💡 Index tickers on Yahoo Finance always start with `^`.


In [ ]:
# Preset index list — ticker: description
PRESET_INDEXES = {
    "^GSPC":    "S&P 500 (USA)",
    "^DJI":     "Dow Jones Industrial Average (USA)",
    "^IXIC":    "NASDAQ Composite (USA)",
    "^RUT":     "Russell 2000 — Small Cap (USA)",
    "^FTSE":    "FTSE 100 (UK)",
    "^GDAXI":   "DAX 40 (Germany)",
    "^FCHI":    "CAC 40 (France)",
    "^STOXX50E":"EURO STOXX 50 (Europe)",
    "^N225":    "Nikkei 225 (Japan)",
    "^HSI":     "Hang Seng (Hong Kong)",
    "^AXJO":    "ASX 200 (Australia)",
    "^BVSP":    "Bovespa (Brazil)",
    "^NSEI":    "Nifty 50 (India)",
    "^KS11":    "KOSPI (South Korea)",
}

preset_options = [f"{ticker}  —  {name}" for ticker, name in PRESET_INDEXES.items()]

preset_select = widgets.SelectMultiple(
    options=preset_options,
    value=["^GSPC  —  S&P 500 (USA)", "^FTSE  —  FTSE 100 (UK)"],
    description="Indexes:",
    layout=widgets.Layout(width="550px", height="280px"),
    style={"description_width": "70px"},
)

custom_input = widgets.Text(
    value="",
    placeholder="e.g.  ^NSEI, ^TSX, ^TA125.TA",
    description="Custom:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="430px"),
)

display(
    widgets.HTML("<b>Select from the list</b> (hold Ctrl or Cmd to select multiple):"),
    preset_select,
    widgets.HTML("<br><b>Add extra tickers</b> (comma-separated, optional):"),
    custom_input,
    widgets.HTML(
        "<small style='color:#666'>Find index tickers at "
        "<a href='https://finance.yahoo.com/world-indices' target='_blank'>"
        "finance.yahoo.com/world-indices</a></small>"
    ),
)


---
## Step 2 — Date Range & Frequency


In [ ]:
start_date = widgets.DatePicker(
    description="Start date:",
    value=datetime.date(2015, 1, 1),
    style={"description_width": "90px"},
)
end_date = widgets.DatePicker(
    description="End date:",
    value=datetime.date.today(),
    style={"description_width": "90px"},
)
frequency = widgets.Dropdown(
    options=["Daily", "Monthly", "Quarterly", "Yearly"],
    value="Monthly",
    description="Frequency:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="220px"),
)

display(
    widgets.HTML("<b>Date range:</b>"),
    widgets.HBox([start_date, end_date]),
    widgets.HTML("<br><b>Frequency:</b>"),
    frequency,
)


---
## Step 3 — Choose Data to Download


In [ ]:
cb_prices     = widgets.Checkbox(value=True,  description="📊  Prices (adjusted close)")
cb_returns    = widgets.Checkbox(value=True,  description="📈  Returns")
cb_cumreturns = widgets.Checkbox(value=False, description="📈  Cumulative Returns")

display(cb_prices, cb_returns, cb_cumreturns)


---
## Step 4 — Download & Save


In [ ]:
output_file = widgets.Text(
    value="market_data.xlsx",
    description="File name:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="320px"),
)

download_btn = widgets.Button(
    description="⬇  Download Data",
    button_style="success",
    layout=widgets.Layout(width="200px", height="40px"),
)

out = widgets.Output()
display(output_file, download_btn, out)


def on_download(b):
    with out:
        clear_output(wait=True)

        # ── Collect tickers ───────────────────────────────────────────────────
        preset_tickers = [s.split("  —  ")[0].strip() for s in preset_select.value]

        custom_raw = custom_input.value.strip()
        custom_tickers = [t.strip().upper() for t in custom_raw.split(",") if t.strip()] if custom_raw else []

        tickers = preset_tickers + custom_tickers

        if not tickers:
            print("❌  Please select at least one index in Step 1.")
            return

        if not cb_prices.value and not cb_returns.value and not cb_cumreturns.value:
            print("❌  Please select at least one data type in Step 3.")
            return

        start = str(start_date.value)
        end   = str(end_date.value)
        freq  = frequency.value

        fname = output_file.value.strip() or "market_data.xlsx"
        if not fname.endswith(".xlsx"):
            fname += ".xlsx"

        output_path = Path("..") / "data" / fname
        sheets = {}

        print(f"Indexes  : {', '.join(tickers)}")
        print(f"Period   : {start}  →  {end}")
        print(f"Frequency: {freq}")
        print()

        print("📊  Downloading prices…")
        try:
            prices = download_prices(tickers, start, end, freq)
            prices = resample_prices(prices, freq)
            print(f"    {len(prices)} rows × {len(prices.columns)} index(es)")

            if cb_prices.value:
                sheets["Index Prices"] = prices

            if cb_returns.value or cb_cumreturns.value:
                returns = calculate_returns(prices)
                if cb_returns.value:
                    sheets["Index Returns"] = returns
                if cb_cumreturns.value:
                    sheets["Index Cum. Returns"] = calculate_cumulative_returns(returns)

            print("    ✅  Done")
        except Exception as e:
            print(f"    ❌  Error: {e}")

        if not sheets:
            print("\n❌  No data collected. Nothing saved.")
            return

        print(f"\n💾  Saving to Excel…")
        try:
            save_to_excel(sheets, output_path)
            print(f"    ✅  Saved: {output_path.resolve()}")
            print(f"    Sheets: {', '.join(sheets.keys())}")
        except Exception as e:
            print(f"    ❌  Could not save: {e}")
            return

        from datetime import datetime as dt
        today = dt.today().strftime("%d %B %Y")
        print()
        print("─" * 60)
        print("📋  DATA SOURCE — copy this into your assignment")
        print("─" * 60)
        print(f"Source     : Yahoo Finance (finance.yahoo.com)")
        print(f"Indexes    : {', '.join(tickers)}")
        print(f"Period     : {start} to {end}  |  Frequency: {freq}")
        print(f"Downloaded : {today}")
        print(f"Tool       : yfinance Python library (pypi.org/project/yfinance)")
        print("─" * 60)


download_btn.on_click(on_download)
